# 6. Воспроизводимый отчёт

**Цель:** выполнить поддерживаемый `GraphEVTPipeline` на Kuramoto-ряде с впрыснутым событием и сохранить численный результат, manifest и краткий отчёт, включая граф, ranking и сверку с известным источником.

In [1]:
from pathlib import Path
from importlib.metadata import version
from hashlib import sha256
import json
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_propagation_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_propagation_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df.columns if column.startswith("channel_")]
x = df[channel_names].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 2

propagation_metadata = json.loads((ROOT / "data/generated/synthetic_datasets_metadata.json").read_text())["kuramoto_propagation"]
event_onset = int(np.flatnonzero(event_mask)[0])
true_source_name = channel_names[int(propagation_metadata["source_channel"])]

In [2]:
BASELINE_SIZE = event_onset
parameters = {
    "baseline_size": BASELINE_SIZE,
    "tail_quantile": 0.85,
    "alarm_probability": 0.97,
    "persistence": 1,
    "top_k": 3,
    "graph_method": "ar",
}
evt = EVTConfig(**{key: value for key, value in parameters.items() if key != "graph_method"})
result = GraphEVTPipeline(
    evt,
    graph=GraphConfig(method=parameters["graph_method"], edge_threshold=0.35, random_state=SEED),
    input_config=InputConfig(missing="error", require_regular_time=True),
    verbose=True,
).run_detailed(x)
detection = result.detection
estimated_source = None if result.ranking is None else channel_names[result.ranking.source]
source_correct = estimated_source == true_source_name

artifact_dir = ROOT / "results/n-d/06_report"
artifact_dir.mkdir(parents=True, exist_ok=True)
indicator_path = artifact_dir / "indicator.csv"
pd.DataFrame({
    "time": df["timestamp"],
    "indicator": detection.indicator,
    "above_alarm_threshold": detection.indicator > detection.alarm_threshold,
    "is_baseline": np.arange(len(x)) < BASELINE_SIZE,
}).to_csv(indicator_path, index=False)

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
manifest = {
    "data_file": str(DATA.relative_to(ROOT)),
    "data_sha256": sha256(DATA.read_bytes()).hexdigest(),
    "library": "graph-evt-agent",
    "library_version": version("graph-evt-agent"),
    "seed": SEED,
    "parameters": parameters,
    "input_profile": {
        "kind": result.input_profile.kind,
        "n_time": result.input_profile.n_time,
        "n_channels": result.input_profile.n_channels,
    },
    "labeled_event": {"onset": event_onset, "source_channel": true_source_name},
    "detection": {
        "detected": detection.detected,
        "time_index": detection.time_index,
        "alarm_threshold": float(detection.alarm_threshold),
    },
    "graph_available": result.graph is not None,
    "estimated_source": estimated_source,
    "source_correct": source_correct,
    "artifacts": {"indicator_csv": indicator_path.name},
    "limitations": [
        "single synthetic multichannel recording, not a benchmark",
        "baseline_size uses the known synthetic onset (oracle-assisted)",
        "graph edges are estimated on the baseline only, not causal",
        "source ranking is an explainable heuristic; correctness here does not generalize",
        "results depend on baseline, EVT configuration, and graph method (see notebook 5)",
    ],
}
with (artifact_dir / "manifest.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

status = "обнаружено" if detection.detected else "не обнаружено"
source_status = "совпал с истинным" if source_correct else "НЕ совпал с истинным"
summary = f"""# Краткий отчёт

На {len(x)} наблюдениях по {result.input_profile.n_channels} каналам устойчивое превышение {status}.
Alarm-порог: {detection.alarm_threshold:.6g}; первый индекс: {detection.time_index}.
Оценённый источник `{estimated_source}` {source_status} истинным источником `{true_source_name}`.
Численный ряд indicator сохранён в `indicator.csv`, параметры и ограничения — в `manifest.json`.
"""
(artifact_dir / "report.md").write_text(summary, encoding="utf-8")
manifest

{'data_file': 'data/generated/kuramoto_propagation_series.csv',
 'data_sha256': 'bcb585f748fc44cf49123587a8ad986b1a343a6b7791390803e2d91cdd5561bc',
 'library': 'graph-evt-agent',
 'library_version': '0.1.0',
 'seed': 42,
 'parameters': {'baseline_size': 932,
  'tail_quantile': 0.85,
  'alarm_probability': 0.97,
  'persistence': 1,
  'top_k': 3,
  'graph_method': 'ar'},
 'input_profile': {'kind': 'multivariate', 'n_time': 1500, 'n_channels': 12},
 'labeled_event': {'onset': 932, 'source_channel': 'channel_00'},
 'detection': {'detected': True,
  'time_index': 949,
  'alarm_threshold': 1.0763861443061515},
 'graph_available': True,
 'estimated_source': 'channel_00',
 'source_correct': True,
 'artifacts': {'indicator_csv': 'indicator.csv'},
 'limitations': ['single synthetic multichannel recording, not a benchmark',
  'baseline_size uses the known synthetic onset (oracle-assisted)',
  'graph edges are estimated on the baseline only, not causal',
  'source ranking is an explainable heurist

In [4]:
required = {"data_sha256", "library_version", "parameters", "input_profile", "detection", "artifacts", "limitations"}
missing = required - set(manifest)
assert not missing, f"Неполный manifest: {sorted(missing)}"
assert len(manifest["data_sha256"]) == 64
summary

'# Краткий отчёт\n\nНа 1500 наблюдениях по 12 каналам устойчивое превышение обнаружено.\nAlarm-порог: 1.07639; первый индекс: 949.\nОценённый источник `channel_00` совпал с истинным истинным источником `channel_00`.\nЧисленный ряд indicator сохранён в `indicator.csv`, параметры и ограничения — в `manifest.json`.\n'

Повторный запуск при той же версии библиотеки и тех же входных байтах должен воспроизводить численный результат, включая граф, ranking и сверку с истинным источником.

**Самопроверка:** почему `source_correct` относится к ограничениям, а не к подтверждённым фактам, даже когда он `True`? Что изменится в manifest, если `--coupling` или `--event-magnitude` генератора будут другими? Какие выводы нельзя переносить за пределы этого конкретного ряда?